[← Back to 02 - SQL Basics](<../02-sql-basics.ipynb>) · [Course Overview](<../00-course-overview.ipynb>)

# 02 - SQL Basics: Solutions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samietex/SQL_Tutorial/blob/main/solutions/02-sql-basics-solutions.ipynb)

Try each exercise in the lesson notebook first. If your query differs from the one here but returns the same rows, it may well be just as correct, and seeing why is a useful exercise in itself.

In [1]:
# Run this cell first. It downloads the small helper that lets the cells below run SQL,
# fetching a fresh copy every time so you always have the latest version.
import os, sys, tempfile, urllib.request
helper_dir = os.path.join(tempfile.gettempdir(), 'sql_tutorial_helper')
os.makedirs(helper_dir, exist_ok=True)
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/samietex/SQL_Tutorial/assets/sql_tutorial.py',
    os.path.join(helper_dir, 'sql_tutorial.py'),
)
sys.path.insert(0, helper_dir)
%reload_ext sql_tutorial

Ready. %%sql runs SQLite 3.50.4 in memory, %%tsql runs SQL Server.


In [2]:
%%sql
CREATE TEMP TABLE lesson_orders (
    order_id INTEGER PRIMARY KEY,
    customer_name TEXT NOT NULL,
    status TEXT NOT NULL,
    total INTEGER
);
INSERT INTO lesson_orders VALUES
    (1, 'Maria', 'completed', 120),
    (2, 'Maria', 'completed', 100),
    (3, 'Maria', 'cancelled', 90),
    (4, 'David', 'completed', 150),
    (5, 'David', 'completed', NULL),
    (6, 'Amina', 'completed', 200),
    (7, 'Amina', 'cancelled', 50);

7 rows inserted


## 1. Predict the rows

In [3]:
%%sql
SELECT
    order_id,
    customer_name,
    total
FROM lesson_orders
WHERE status = 'completed' AND total < 150;

order_id,customer_name,total
1,Maria,120
2,Maria,100


Two orders, both Maria's: order 1 at 120 and order 2 at 100. David's order 4 is exactly 150, and 150 isn't less than 150. Amina's 200 is too large. And David's order 5 is completed, but its missing total makes `total < 150` unknown, so it's left out, the same way it was left out of both sides of the $100 filter. If you said three orders, you most likely counted order 4 (`<=` would include it) or order 5.

## 2. David's and Amina's big orders

In [4]:
%%sql
SELECT
    order_id,
    customer_name,
    total
FROM lesson_orders
WHERE status = 'completed'
  AND customer_name IN ('David', 'Amina')
  AND total >= 150
ORDER BY
    total DESC;

order_id,customer_name,total
6,Amina,200
4,David,150


Amina's 200 and David's 150. "At least 150" means `>=`, so order 4 is in. `IN` keeps the two names as one condition, so there's no `AND`/`OR` precedence to get wrong; if you wrote `customer_name = 'David' OR customer_name = 'Amina'` instead, it needs parentheses around it to give the same answer.

## 3. Orders that need a look

In [5]:
%%sql
SELECT
    order_id,
    customer_name,
    status,
    total
FROM lesson_orders
WHERE status <> 'completed' OR total IS NULL
ORDER BY
    order_id;

order_id,customer_name,status,total
3,Maria,cancelled,90
5,David,completed,NULL
7,Amina,cancelled,50


Orders 3, 5, and 7: the two cancelled orders and David's order with no total. `total = NULL` would have found nothing, so order 5 needs `IS NULL`. There's only one kind of operator here, `OR`, so no parentheses are needed, though adding them never hurts.

## 4. Stretch: black products (AdventureWorks)

This answer runs on SQL Server, so set up AdventureWorks first. In Colab that takes a few minutes the first time, and on your own machine it only checks the connection.

In [6]:
# Run this once before any %%tsql cell.
# In Google Colab it installs SQL Server and AdventureWorks into this session (about 3 to 5
# minutes the first time, and again in each new session, because Colab starts fresh).
# On your own machine it just checks that your SQL Server is reachable.
%setup_adventureworks

AdventureWorks2022 is ready, so every %%tsql cell in this notebook will run.


In [7]:
%%tsql
SELECT COUNT(*) AS black_products
FROM Production.Product
WHERE Color = 'Black';

black_products
93


In [8]:
%%tsql
SELECT COUNT(*) AS not_black_products
FROM Production.Product
WHERE Color <> 'Black';

not_black_products
163


93 plus 163 is 256, so 248 products are on neither side, which is nearly half the catalog. They're products with no color recorded:

In [9]:
%%tsql
SELECT COUNT(*) AS no_color
FROM Production.Product
WHERE Color IS NULL;

no_color
248


93 + 163 + 248 = 504. Most of them, 198 of the 248, are components rather than finished products, things like lock nuts, washers, and spokes, where a color doesn't apply, which is a good reminder that `NULL` can mean "doesn't apply" as well as "nobody recorded it", and the query can't tell you which; only knowing the data can.

---

[← Back to 02 - SQL Basics](<../02-sql-basics.ipynb>) · [Course Overview](<../00-course-overview.ipynb>)

*SQL_Tutorial* is written and maintained by Samuel Shaibu as part of *All About Data & More*. Licensed under [MIT](../LICENSE).